# RAG Cheat Sheet — Practical Companion
Based on the six screenshots supplied by PaulRobertH. Work through this alongside **RAG with Pytorch-v1.ipynb**.

Text file → passages → DPR context embeddings → FAISS index → DPR question embedding → retrieved passages → GPT-2 completion.

This notebook explains each component and includes a small fictional company policy dataset. It does not train a model. Run in a Python 3.11 Codespace kernel. Model cells download two DPR models and GPT-2, so allow time and memory for downloads. No API key is required. GPT-2 is a base language model, so retrieved context does not guarantee a correct answer.


## 1. Install packages
Run this cell once, then restart the kernel if prompted. The Transformers 4.x range keeps the teaching API consistent; this dependency combination has not been installed or tested here.


In [ ]:
%pip install "transformers>=4.42,<5" torch numpy faiss-cpu scikit-learn matplotlib

In [ ]:
from pathlib import Path
import numpy as np
import torch
import faiss
from transformers import (DPRContextEncoder, DPRContextEncoderTokenizer,
                          DPRQuestionEncoder, DPRQuestionEncoderTokenizer,
                          AutoTokenizer, AutoModelForCausalLM)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 2. Read and split text
The screenshot splits on every newline: it produces non-empty lines, not necessarily full paragraphs. We preserve that behaviour in `read_and_split_text`. For real documents, consider paragraph-aware or token-sized chunks. Longer lines will be truncated by the encoder below.


In [ ]:
def read_and_split_text(filename):
    return [line.strip() for line in Path(filename).read_text(encoding='utf-8').splitlines() if line.strip()]

sample = Path('rag_companion_demo_policies.txt')
if not sample.exists():
    sample.write_text(
        'Fictional demo policy: Employees receive 25 days of annual leave.\n'
        'Fictional demo policy: Remote working is allowed up to two days each week.\n'
        'Fictional demo policy: Expense claims must be submitted within 30 days.\n'
        'Fictional demo policy: Office hours are 9 am to 5 pm, Monday to Friday.\n'
        'Fictional demo policy: IT support can be reached at helpdesk@example.com.\n'
        'Fictional demo policy: Training requests require manager approval.\n', encoding='utf-8')
paragraphs = read_and_split_text(sample)
assert paragraphs, 'Add at least one non-empty passage.'
for i, text in enumerate(paragraphs):
    print(i, text)


## 3. Load the paired DPR encoders
The context tokenizer converts text into model inputs; it does not create the embedding itself. The context encoder creates passage vectors. The matching question encoder creates query vectors. Both outputs use `pooler_output`. [DPR documentation](https://huggingface.co/docs/transformers/model_doc/dpr).


In [ ]:
context_name = 'facebook/dpr-ctx_encoder-single-nq-base'
question_name = 'facebook/dpr-question_encoder-single-nq-base'
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained(context_name)
context_encoder = DPRContextEncoder.from_pretrained(context_name).to(device).eval()
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(question_name)
question_encoder = DPRQuestionEncoder.from_pretrained(question_name).to(device).eval()


In [ ]:
def encode_contexts(text_list, batch_size=4):
    if not text_list:
        raise ValueError('No passages to encode.')
    if batch_size < 1:
        raise ValueError('batch_size must be positive.')
    embeddings = []
    with torch.inference_mode():
        for start in range(0, len(text_list), batch_size):
            inputs = context_tokenizer(text_list[start:start+batch_size],
                return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
            embeddings.append(context_encoder(**inputs).pooler_output.cpu())
    return np.ascontiguousarray(torch.cat(embeddings).numpy(), dtype='float32')

context_embeddings = encode_contexts(paragraphs)
print('Shape (passages, dimensions):', context_embeddings.shape)


## 4. Build FAISS indexes
`IndexFlatL2` returns **squared** Euclidean distances; smaller values are closer. The original DPR retriever uses dot products, so we also build `IndexFlatIP` and use it by default. Larger inner-product scores rank first. Derive the vector dimension from the array instead of hard-coding 768. Scores are not probabilities. [FAISS metric guide](https://github.com/facebookresearch/faiss/wiki/MetricType-and-distances).


In [ ]:
dimension = context_embeddings.shape[1]
index_l2 = faiss.IndexFlatL2(dimension)
index_l2.add(context_embeddings)
index_ip = faiss.IndexFlatIP(dimension)
index_ip.add(context_embeddings)
index = index_ip
print('Indexed passages:', index.ntotal)


In [ ]:
def search_relevant_contexts(question, question_tokenizer, question_encoder, index, k=5):
    if not question.strip():
        raise ValueError('Enter a non-empty question.')
    if index.ntotal == 0 or k < 1:
        raise ValueError('Use a non-empty index and positive k.')
    encoder_device = next(question_encoder.parameters()).device
    with torch.inference_mode():
        inputs = question_tokenizer(question, return_tensors='pt', truncation=True,
                                    max_length=512).to(encoder_device)
        vector = question_encoder(**inputs).pooler_output.cpu().numpy()
    vector = np.ascontiguousarray(vector, dtype='float32')
    return index.search(vector, min(k, index.ntotal))

question = 'How many days of annual leave do employees receive?'
scores, indices = search_relevant_contexts(question, question_tokenizer, question_encoder, index, k=3)
for score, i in zip(scores[0], indices[0]):
    print(f'Passage {i}, score {score:.3f}: {paragraphs[i]}')
print('Inspect whether the retrieved passages actually answer the question.')


## 5. Visualise embeddings with t-SNE
t-SNE is an exploratory projection. Distances and clusters in the plot do not prove retrieval quality. Unlike the screenshot, this helper handles small datasets and sets a valid perplexity. Colours identify passage indexes, not known topic classes.


In [ ]:
def tsne_plot(data):
    import matplotlib.pyplot as plt
    from sklearn.manifold import TSNE
    data = np.asarray(data)
    if data.ndim != 2 or len(data) < 4:
        print('Use at least four embedding vectors for this 3D plot.')
        return
    points = TSNE(n_components=3, init='random', random_state=42,
                  perplexity=min(30, len(data)-1), learning_rate='auto').fit_transform(data)
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(*points.T, c=np.arange(len(points)), cmap='tab20')
    for i, point in enumerate(points):
        ax.text(*point, str(i))
    ax.set(xlabel='t-SNE 1', ylabel='t-SNE 2', zlabel='t-SNE 3', title='Passage embeddings (exploratory)')
    plt.show()

tsne_plot(context_embeddings)


## 6. Load GPT-2 and compare completions
GPT-2 predicts continuations; it was not instruction-tuned to answer policy questions. Use this to observe the difference between retrieval and generation. Check each generated claim against the printed passages.

We set an EOS-based padding token, pass the attention mask, reserve space for new tokens, and decode only the continuation. These changes avoid confusing prompt length with output length.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2').to(device).eval()
tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.pad_token_id

def complete_prompt(prompt, max_new_tokens=80):
    limit = model.config.n_positions
    if not 0 < max_new_tokens < limit:
        raise ValueError('max_new_tokens must fit inside the model context window.')
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=limit-max_new_tokens).to(device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def generate_answer_without_context(question):
    return complete_prompt(f'Question: {question}\nAnswer:')

def generate_answer_with_context(question, k=3):
    scores, indices = search_relevant_contexts(question, question_tokenizer, question_encoder, index, k)
    sources = [(int(i), paragraphs[i]) for i in indices[0]]
    context = '\n'.join(f'[{i}] {text}' for i, text in sources)
    # Cap context separately so a long source cannot truncate the question.
    context_tokens = tokenizer.encode(context, add_special_tokens=False)[:700]
    context = tokenizer.decode(context_tokens)
    prompt = f'Use the policy excerpts below to answer the question. If unsupported, say you do not know.\n{context}\nQuestion: {question}\nAnswer:'
    return complete_prompt(prompt), sources

print('WITHOUT CONTEXT:', generate_answer_without_context(question))
answer, sources = generate_answer_with_context(question)
print('WITH CONTEXT:', answer)
print('RETRIEVED SOURCES:', sources)


## 7. Evaluate and experiment
Try questions about remote working and expense deadlines. Then ask “What is the company pension contribution?”—the sample contains no answer. Check separately:

1. Did retrieval find an answer-bearing passage?
2. Did generation use that passage accurately?
3. Did the response invent facts when evidence was missing?

Compare `index_ip` with `index_l2` by assigning either to `index`. Record which metric retrieves the correct passage. Replace the fictional text file with your own non-sensitive document and re-run encoding and indexing.

## Troubleshooting and validation
If imports fail, check the selected kernel matches the environment where packages were installed. If memory is tight, use smaller batches and CPU; the two encoders and generator remain in memory together. If results are poor, inspect chunks and source relevance before changing generation settings.

Created from the supplied screenshots with newly written explanations and sample policies. Notebook schema and Python syntax were validated locally. The text-splitting helper was exercised on a fixture. Model downloads, DPR retrieval, FAISS execution, t-SNE, and GPT-2 generation have not been run during creation; measure their behaviour in your Codespace.
